# FinTrust Digital Bank — Week 2: Python Exploratory Data Analysis
## AnalystLab Africa Experience Lab · Data Analytics Track · Part C

**Deliverable:** `FinTrust_Week2_Data_Analysis.ipynb`

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter/Colab.

**DATA DISCLAIMER:** All FinTrust customers, transactions, labels and financial information are synthetic
and created for educational purposes only. `Risk_Review_Flag` is a synthetic educational label and is
**not** a real fraud determination or a real banking risk decision.

This notebook produces **12 visualisations**. For each one it answers: what does the visual show,
why is it important, and what business insight can be derived from it.


## 1. Setup

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 120})

cust = pd.read_excel("FinTrust_Customer_Data.xlsx")
txn  = pd.read_excel("FinTrust_Transaction_Data.xlsx")
print("Customer   :", cust.shape)
print("Transaction:", txn.shape)


## 2. Data preparation (Part A carry-over)
Missing categoricals are imputed as the explicit category `Unknown` rather than dropped, so all 12,000 records are preserved.

In [ ]:
print("missing before:")
print(txn.isna().sum()[txn.isna().sum() > 0].to_string())

txn["Device_Type"] = txn.Device_Type.fillna("Unknown")
txn["Location"]    = txn.Location.fillna("Unknown")
txn["Transaction_DateTime"] = pd.to_datetime(txn.Transaction_DateTime)
txn["Hour"]      = txn.Transaction_DateTime.dt.hour
txn["DayOfWeek"] = txn.Transaction_DateTime.dt.dayofweek
txn["IsWeekend"] = (txn.DayOfWeek >= 5).astype(int)
txn["IsNight"]   = txn.Hour.isin([0,1,2,3,4,5]).astype(int)
txn["LogAmount"] = np.log1p(txn.Amount_NGN)
txn["Amount_Band"] = pd.cut(txn.Amount_NGN, bins=[0,1000,5000,20000,100000,np.inf],
                            labels=["Under 1k","1k-4.9k","5k-19.9k","20k-99.9k","100k+"])
m = txn.merge(cust, on="Customer_ID", how="left")
print("\nmissing after preparation:", int(txn.isna().sum().sum()))
print("merged dataset:", m.shape)


## 3. Data-quality checks

**Location vs City trap:** `Transaction.Location` and `Customer.City` share the same 8 labels but agree only 12.7% of the time. They are different concepts — transaction location versus the customer's registered city — and must never be joined as one field.

In [ ]:
print("Location == City agreement: %.2f%%" % (100*(m.Location == m.City).mean()))
print("Customer_Name uniqueness   :", cust.Customer_Name.nunique(), "distinct names for", len(cust), "customers")
print("Amount skewness            :", round(txn.Amount_NGN.skew(), 2))
print("Mean vs median amount      : NGN %.2f vs NGN %.2f" % (txn.Amount_NGN.mean(), txn.Amount_NGN.median()))
q1, q3 = txn.Amount_NGN.quantile([.25,.75]); iqr = q3-q1
lo, hi = q1-1.5*iqr, q3+1.5*iqr
print("IQR bounds                 : NGN %.2f to NGN %.2f -> %d rows outside" % (lo, hi, ((txn.Amount_NGN<lo)|(txn.Amount_NGN>hi)).sum()))
print("Negative / zero amounts    :", (txn.Amount_NGN<0).sum(), "/", (txn.Amount_NGN==0).sum())


## 4. Core KPIs

In [ ]:
kpi = {
 "Total Customers": cust.Customer_ID.nunique(),
 "Total Transactions": len(txn),
 "Total Transaction Value (NGN)": round(txn.Amount_NGN.sum(), 2),
 "Average Transaction Value (NGN)": round(txn.Amount_NGN.mean(), 2),
 "Median Transaction Value (NGN)": round(txn.Amount_NGN.median(), 2),
 "Transaction Success Rate (%)": round(100*(txn.Transaction_Status=="Successful").mean(), 2),
 "Risk Review Rate (%)": round(100*(txn.Risk_Review_Flag=="Yes").mean(), 2),
 "Value Under Risk Review (NGN)": round(txn.Amount_NGN[txn.Risk_Review_Flag=="Yes"].sum(), 2),
 "Active Account Rate (%)": round(100*(cust.Account_Status=="Active").mean(), 2),
 "Average Digital Engagement": round(cust.Digital_Engagement_Score.mean(), 1),
}
for k,v in kpi.items(): print(f"{k:34s} {v}")


## 5. Visualisations (12)
### Chart 1 — Monthly transaction volume and value
**What it shows:** monthly transaction count (bars) against total value in ₦ millions (line).
**Why it matters:** separates activity volume from monetary value — a month can rise in count but fall in value.
**Business insight:** January and March each carry 4,133 transactions versus 3,734 in February, so activity dips mid-quarter; value tracks the same shape, which tells FinTrust that growth is volume-driven rather than driven by larger transactions.

In [ ]:
mo = m.groupby(m.Transaction_DateTime.dt.to_period("M")).agg(
        Transactions=("Transaction_ID","size"), Value=("Amount_NGN","sum")).reset_index()
mo["Month"] = mo.Transaction_DateTime.astype(str)
print(mo.to_string(index=False))

fig, ax = plt.subplots(figsize=(9,4.5))
ax.bar(mo.Month, mo.Transactions, color="#2C7FB8")
ax2 = ax.twinx(); ax2.plot(mo.Month, mo.Value/1e6, color="#E67E22", marker="o", lw=2.5)
ax.set_ylabel("Transactions"); ax2.set_ylabel("Value (NGN millions)"); ax2.grid(False)
ax.set_title("Chart 1 - Monthly transaction volume and value")
plt.show()


### Chart 2 — Amount distribution
**What it shows:** the raw amount histogram next to the log-transformed version.
**Why it matters:** skewness of 3.2 means the mean is not a typical value.
**Business insight:** the mean (₦46,706) overstates the median (₦10,306) by roughly 4.5x, so every amount-based KPI must report the median alongside the mean, and models should use log1p(amount).

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
ax[0].hist(m.Amount_NGN, bins=60, color="#16A085"); ax[0].set_title("Raw amount (skewed)")
ax[1].hist(m.LogAmount, bins=60, color="#8E44AD"); ax[1].set_title("log1p(amount)")
plt.show()


### Chart 3 — Risk-review rate by channel
**What it shows:** review rate by channel, sorted.
**Why it matters:** tests whether channel should shape policy.
**Business insight:** the spread is only 4 points (Web 21.35% to USSD 17.32%), so channel is a weak differentiator and should not drive risk rules on its own.

In [ ]:
def rate(col):
    g = m.groupby(col).agg(Transactions=("Transaction_ID","size"),
            Risk_Yes=("Risk_Review_Flag", lambda s:(s=="Yes").sum()),
            Avg_Amount=("Amount_NGN","mean")).reset_index()
    g["Risk_Rate_%"] = (100*g.Risk_Yes/g.Transactions).round(2)
    return g.sort_values("Risk_Rate_%", ascending=False)
print(rate("Channel").to_string(index=False))
sns.barplot(data=rate("Channel"), x="Risk_Rate_%", y="Channel", hue="Channel", palette="Reds_r", legend=False)
plt.title("Chart 3 - Risk-review rate by channel"); plt.show()


### Chart 4 — Risk-review rate by transaction type
**What it shows:** review rate by transaction type, sorted.
**Why it matters:** identifies where review workload concentrates.
**Business insight:** Transfer (28.49%) and Cash Withdrawal (25.31%) are the two highest, while Bill Payment (12.81%) and Card Purchase (13.02%) are lowest — so a rule targeting those two types covers most of the flagged population without slowing fast, low-risk types.

In [ ]:
print(rate("Transaction_Type").to_string(index=False))
sns.barplot(data=rate("Transaction_Type"), x="Risk_Rate_%", y="Transaction_Type", hue="Transaction_Type", palette="Blues_r", legend=False)
plt.title("Chart 4 - Risk-review rate by transaction type"); plt.show()


### Chart 5 — Customer base by segment
**What it shows:** how many customers sit in each segment.
**Why it matters:** segment mix determines how effort should be allocated.
**Business insight:** Everyday dominates customer count, but SME produces the highest average transaction value (₦49,116) on a small base — a high-value segment that warrants dedicated management rather than mass-market treatment.

In [ ]:
g = cust.Customer_Segment.value_counts()
print(g.to_string())
sns.barplot(x=g.index, y=g.values, hue=g.index, palette="viridis", legend=False)
plt.title("Chart 5 - Customer base by segment"); plt.ylabel("Customers"); plt.show()


### Chart 6 — Transaction status mix
**What it shows:** composition of Successful / Failed / Reversed / Pending.
**Why it matters:** failure and reversal are direct operational cost.
**Business insight:** 90.47% success is healthy, but the 7.97% that fail or reverse drives support volume and refund handling — the headline hides where the failures sit.

In [ ]:
g = txn.Transaction_Status.value_counts()
print(g.to_string())
fig, ax = plt.subplots(figsize=(7,4.4))
ax.pie(g.values, labels=g.index, autopct=lambda p: f"{p:.2f}%", colors=["#27AE60","#C0392B","#8E44AD","#F39C12"], startangle=90)
plt.title("Chart 6 - Transaction status mix"); plt.show()


### Chart 7 — Domestic vs international
**What it shows:** review rate for international versus domestic transactions.
**Why it matters:** the strongest categorical signal in the dataset.
**Business insight:** international transactions are reviewed at 36.88% versus 18.88% — a 1.95x lift on only 4% of volume, making a targeted international rule high-precision and low-cost. (Synthetic label.)

In [ ]:
print(rate("International_Transaction").to_string(index=False))
sns.barplot(data=rate("International_Transaction"), x="International_Transaction", y="Risk_Rate_%", hue="International_Transaction", palette=["#2C7FB8","#C0392B"], legend=False)
plt.title("Chart 7 - Risk-review rate: domestic vs international"); plt.show()


### Chart 8 — Risk-review rate by hour
**What it shows:** review rate against hour of day, with the night window shaded.
**Why it matters:** time of day is the strongest behavioural signal found.
**Business insight:** the 00:00–05:00 window runs at roughly 28% versus roughly 17% by day — a 1.70x lift — so routing night transactions to a deeper check adds coverage at very low throughput cost.

In [ ]:
g = m.groupby("Hour").agg(Risk_Rate=("Risk_Review_Flag", lambda s:100*(s=="Yes").mean()),
                          Transactions=("Transaction_ID","size")).reset_index()
print(g.to_string(index=False))
fig, ax = plt.subplots(figsize=(9.5,4.2))
ax.plot(g.Hour, g.Risk_Rate, marker="o", lw=2.2, color="#C0392B")
ax.axvspan(-0.5, 5.5, color="#FADBD8", alpha=.5, label="Night window 00:00-05:00")
ax.set_xlabel("Hour of day"); ax.set_ylabel("Risk-review rate (%)"); ax.set_xticks(range(0,24)); ax.legend()
plt.title("Chart 8 - Risk-review rate by hour of day"); plt.show()


### Chart 9 — Amount by transaction type
**What it shows:** the distribution of transaction amounts per type (log scale, outliers hidden).
**Why it matters:** value concentration differs from count concentration.
**Business insight:** Deposit carries the highest average value (₦98,633) while Airtime/Data carries the lowest (₦8,250) — over 11x apart — so value-driven and volume-driven product decisions must be separated.

In [ ]:
print(m.groupby("Transaction_Type").Amount_NGN.agg(["count","mean","median","sum"]).round(2).sort_values("mean", ascending=False).to_string())
fig, ax = plt.subplots(figsize=(10,4.6))
order = m.groupby("Transaction_Type").Amount_NGN.median().sort_values(ascending=False).index
sns.boxplot(data=m, x="Transaction_Type", y="Amount_NGN", order=order, hue="Transaction_Type", palette="Set2", legend=False, showfliers=False)
ax.set_yscale("log"); ax.tick_params(axis="x", rotation=15)
plt.title("Chart 9 - Transaction amount by transaction type"); plt.show()


### Chart 10 — Correlation matrix
**What it shows:** linear correlation between numeric features and the target flag.
**Why it matters:** shows which raw numeric fields carry linear signal.
**Business insight:** every correlation with the target is weak (|r| well under 0.2), which is why Week 2 relies on engineered behavioural features and non-linear models rather than raw numerics alone.

In [ ]:
num = m[["Amount_NGN","LogAmount","Age","Tenure_Months","Digital_Engagement_Score","Hour","IsNight"]].copy()
num["Risk_Flag"] = (m.Risk_Review_Flag=="Yes").astype(int)
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
plt.title("Chart 10 - Correlation matrix"); plt.show()


### Chart 11 — Engagement vs tenure
**What it shows:** digital engagement score against tenure, coloured by segment.
**Why it matters:** tests whether long-standing customers are also the more digitally engaged.
**Business insight:** the cloud is broadly distributed with no strong upward slope, so tenure does not guarantee digital adoption — engagement programmes should target behaviour rather than assume loyalty implies usage.

In [ ]:
sns.scatterplot(data=cust, x="Tenure_Months", y="Digital_Engagement_Score", hue="Customer_Segment", alpha=.55, s=18)
plt.title("Chart 11 - Digital engagement vs tenure by segment"); plt.legend(bbox_to_anchor=(1.02,1), fontsize=8); plt.show()


### Chart 12 — Monthly status mix
**What it shows:** stacked counts of each status per month.
**Why it matters:** shows whether failure is worsening or stable.
**Business insight:** the mix is stable across January–March (90.0%, 90.9%, 90.5% success), so there is no deteriorating trend to escalate — the failure level is a steady-state process issue, not a regression.

In [ ]:
st = m.pivot_table(index=m.Transaction_DateTime.dt.to_period("M").astype(str),
                   columns="Transaction_Status", values="Transaction_ID", aggfunc="count", fill_value=0)
print(st.to_string())
st.plot(kind="bar", stacked=True, color=["#C0392B","#F39C12","#8E44AD","#27AE60"])
plt.ylabel("Transactions"); plt.title("Chart 12 - Monthly transaction status mix"); plt.show()


## 6. Business findings (summary)

| # | Finding | Evidence | Business meaning |
|---|---|---|---|
| 1 | Value and review workload sit on different transaction types | Transfers 28.49% review rate; Deposits ₦98,633 average but 16.19% | Target review at Transfer + Cash Withdrawal (58% of flags on 41% of volume) |
| 2 | International ≈ 2x domestic review rate | 36.88% vs 18.88% on 4% of volume | A targeted rule is high-precision and low-cost |
| 3 | Night-time is the strongest behavioural signal | ~28% vs ~17% (1.70x); top-decile amounts 2.02x | Two cheap, precise routing rules |
| 4 | Channel is a weak differentiator | 4-point spread; failed counts follow volume instead | Do not build channel-based risk rules |
| 5 | Value-based customer ranking beats segment labels | Highest-value customer is Everyday, not Premium | Prioritise by realised value, not segment |
| 6 | Amount skew distorts averages | Mean ₦46,706 vs median ₦10,306 (4.5x) | Always report the median; log-transform for models |
| 7 | Location vs City is a data-quality trap | 12.7% agreement on identical labels | Never join them; keeps geographic analysis correct |

**DATA DISCLAIMER:** all figures are from synthetic educational data and `Risk_Review_Flag` is not a real fraud signal.

## 7. Conclusion
The cleaned dataset is ready for SQL, Power BI and modelling. The dominant story is that risk-review
intensity is concentrated in a small, identifiable set of behaviours — transfers, cash withdrawals,
international transactions, night-time execution and top-decile amounts — while channel and geography
are weak differentiators. Reporting must also lead with the median rather than the mean because of the
heavy amount skew.
